In [1]:
platformID = 'INS'

In [2]:
from datetime import datetime
import pandas as pd
import numpy as np
import os 

import psycopg2


## import helper 

In [3]:
import sys
from pathlib import Path

try:
    # Works in Python scripts
    helper_path = Path(__file__).resolve().parent.parent / "helper"
except NameError:
    # Works in Jupyter notebooks
    helper_path = Path().resolve().parent / "helper"

sys.path.insert(0, str(helper_path))

# Now import your modules 
from config import gam_info
import test_functions 

In [4]:
# country
country_codes = pd.read_excel(f"../../{gam_info['lookup_file']}", sheet_name='CountryID', 
                              keep_default_na=False)

# week 
week_tester = pd.read_excel(f"../../{gam_info['lookup_file']}", sheet_name='GAM Period')
week_tester['w/c'] = pd.to_datetime(week_tester['w/c'])

# social media accounts
socialmedia_accounts = pd.read_excel("../helper/ins_account_lookup.xlsx")


# ingest 

In [5]:
engagements = pd.read_csv(f"../data/processed/{platformID}/{gam_info['file_timeinfo']}_{platformID}_engagements_final.csv",)
engagements.columns

Index(['Channel ID', 'Channel Name', 'ig_user_linked_fb_page_id',
       'IG Account URL', 'w/c', 'ServiceID', 'IG studios exc uk',
       'weekly_media_engagements', 'saved', 'plays', 'daily_avg_reach',
       'reach', 'impressions', '30 view', 'IG Modelled Factor',
       'IG Engaged Users', 'IG Engaged Persian Exception'],
      dtype='object')

In [20]:
# can't find this channel 17841402094893665
engagements['Channel ID'].sort_values().unique()

array([17841400072681315, 17841400230391592, 17841400240564037,
       17841400273561016, 17841400323746756, 17841400485756476,
       17841400546813714, 17841400559646664, 17841400581363643,
       17841400630063846, 17841400681780254, 17841400778089100,
       17841400884455643, 17841400948656588, 17841401010530114,
       17841401143954624, 17841401184448004, 17841401322662359,
       17841401325222869, 17841401350693298, 17841401471775462,
       17841401518836664, 17841401562994997, 17841401580814335,
       17841401606325704, 17841401623832118, 17841401635876674,
       17841401637906903, 17841401682931831, 17841401718394152,
       17841401898135827, 17841401970502619, 17841402094538514,
       17841402172935746, 17841402180741995, 17841402244137767,
       17841402263867175, 17841402309448205, 17841402326929599,
       17841402342979604, 17841402344872147, 17841402346743687,
       17841402367900365, 17841402374279647, 17841402949055863,
       17841403782551891, 17841404661114

In [6]:
country = pd.read_csv(f"../data/processed/{platformID}/{gam_info['file_timeinfo']}_{platformID}_REDSHIFT_geog.csv",
                     keep_default_na=False)
country.columns

Index(['Channel ID', 'Channel Name', 'ig_metric_id', 'ig_metric_period',
       'ig_metric_breakdown', 'week_ending', 'ig_metric_value',
       'Sum_ig_metric_value', 'country_%', 'w/c', 'ServiceName'],
      dtype='object')

In [19]:
# can't find this channel 17841402094893665
country['Channel ID'].sort_values().unique()

array([17841400072681315, 17841400230391592, 17841400240564037,
       17841400273561016, 17841400485756476, 17841400546813714,
       17841400559646664, 17841400581363643, 17841400630063846,
       17841400681780254, 17841400778089100, 17841400884455643,
       17841400948656588, 17841401010530114, 17841401143954624,
       17841401184448004, 17841401322662359, 17841401325222869,
       17841401350693298, 17841401471775462, 17841401518836664,
       17841401562994997, 17841401580814335, 17841401606325704,
       17841401623832118, 17841401635876674, 17841401637906903,
       17841401682931831, 17841401718394152, 17841401898135827,
       17841402094538514, 17841402172935746, 17841402180741995,
       17841402244137767, 17841402263867175, 17841402309448205,
       17841402326929599, 17841402342979604, 17841402344872147,
       17841402346743687, 17841402367900365, 17841402374279647,
       17841402949055863, 17841404661114773, 17841404663664767,
       17841404702394750, 17841404704674

In [7]:
country_annual_avg = country.groupby(['Channel ID', 'Channel Name', 
                                      'ig_metric_breakdown'])['country_%'].mean().reset_index()


# combine 

In [8]:
combined = engagements.merge(country, on=["Channel ID", "w/c"], how='left', indicator=True)
combined_inner = combined[combined['_merge'] == 'both'].drop(columns='_merge')
combined_left = combined[combined['_merge'] == 'left_only'].drop(columns='_merge')

left_matched = combined_left.merge(country_annual_avg, on="Channel ID", how='left', indicator=True)

cols_to_clean = ['Channel Name', 'ig_metric_breakdown', 'country_%',]
for col in cols_to_clean:
    left_matched[f"{col}"] = left_matched[f"{col}_x"].fillna(left_matched[f"{col}_y"])
    left_matched = left_matched.drop(columns=[f"{col}_x", f"{col}_y"])

temp = left_matched[left_matched['_merge'] == 'left_only'].drop(columns='_merge')
temp = temp.merge(socialmedia_accounts[['Channel ID', 'IG Handle']], on='Channel ID', how='left')
missing_country_perc = pd.read_excel("../../../../Research Projects/GAM/Digital GAM/2025/Social Media/data/missing ig countries.xlsx")
missing_country_perc['country code'] = missing_country_perc['country code'].str.upper()
missing_country_perc.rename(columns={'country code': 'ig_metric_breakdown', 'Total': 'country_%'})
temp = temp.merge(missing_country_perc, on='IG Handle', how='inner')
cols = ['Channel ID', 'Channel Name', 'IG Handle', 'ig_user_linked_fb_page_id', 
        'w/c', 'IG Account URL', 'ServiceID', 'IG studios exc uk',
        'weekly_media_engagements', 'saved', 'plays', 'daily_avg_reach', 'reach', 'impressions',
        '30 view', 'IG Modelled Factor', 'IG Engaged Users', 'IG Engaged Persian Exception',
        'ig_metric_breakdown', 'country_%']
temp = temp[cols]

/var/folders/gz/pq5c3fbj5rs1tz_5w1hycq4h0000gn/T/ipykernel_3940/3535252636.py:9: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  left_matched[f"{col}"] = left_matched[f"{col}_x"].fillna(left_matched[f"{col}_y"])


In [9]:

cols = ['Channel ID', 'Channel Name', 'IG Handle', 'ig_user_linked_fb_page_id',
       'w/c', 'IG Account URL', 'ServiceID', 'IG studios exc uk',
       'plays', 'daily_avg_reach', 'IG Engaged Persian Exception', 
        'ig_metric_breakdown', 'country_%']
engagement_country = pd.concat([combined_inner, temp])[cols].rename(columns={'IG Engaged Persian Exception': 'IG Engaged Users'})



In [10]:
to_clean_country = country_codes[['PlaceID', 'YT-_FBE_codes', gam_info['population_column']]].rename(columns={'YT-_FBE_codes': 'ig_metric_breakdown'})
clean_country = engagement_country.merge(to_clean_country, on='ig_metric_breakdown', how='left', indicator=True)
print(clean_country.shape)
final_ig_data = clean_country.drop_duplicates(subset=['PlaceID', 'w/c', gam_info['population_column'],
                                                      'Channel ID', 'Channel Name', 'IG Account URL'])
print(final_ig_data.shape)
final_ig_data['IG Engaged Users'] = final_ig_data['IG Engaged Users'].fillna(0)
final_ig_data['country_%'] = final_ig_data['country_%'].fillna(0)
final_ig_data['uv_by_country'] = final_ig_data['IG Engaged Users'] * final_ig_data['country_%']

(257829, 16)
(171474, 16)


/var/folders/gz/pq5c3fbj5rs1tz_5w1hycq4h0000gn/T/ipykernel_3940/2443167330.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_ig_data['IG Engaged Users'] = final_ig_data['IG Engaged Users'].fillna(0)
/var/folders/gz/pq5c3fbj5rs1tz_5w1hycq4h0000gn/T/ipykernel_3940/2443167330.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_ig_data['country_%'] = final_ig_data['country_%'].fillna(0)
/var/folders/gz/pq5c3fbj5rs1tz_5w1hycq4h0000gn/T/ipykernel_3940/2443167330.py:9: SettingWithCopyWarning: 
A valu

# store 

In [11]:
cols = ['w/c', 'PlaceID', 'ServiceID', 'Channel ID', 'uv_by_country']
final_ig_data[cols].to_csv(f"../data/processed/{platformID}/{gam_info['file_timeinfo']}_uniqueViewer_country.csv",
                     index=None)


In [13]:
final_ig_data[final_ig_data['Channel ID'] == 17841402094893665]

,Channel ID,Channel Name,IG Handle,ig_user_linked_fb_page_id,w/c,IG Account URL,ServiceID,IG studios exc uk,plays,daily_avg_reach,IG Engaged Users,ig_metric_breakdown,country_%,PlaceID,Population2020,_merge,uv_by_country
